# 01 — Data Preparation

Convert the transaction table into an account-centric long format and create
the train/test account reference table used by later notebooks.

> Competition data is intentionally not included in this repository.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore", category=FutureWarning)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = REPO_ROOT / "data"
ARTIFACT_DIR = REPO_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TRANSACTION_FILE = DATA_DIR / "acct_transaction.csv"
ALERT_FILE = DATA_DIR / "acct_alert.csv"
PREDICT_FILE = DATA_DIR / "acct_predict.csv"

In [ ]:
# Load competition tables.
df_txn = pd.read_csv(TRANSACTION_FILE, dtype={"txn_date": "Int64"})
df_alert = pd.read_csv(ALERT_FILE, dtype={"event_date": "Int64"})
df_predict = pd.read_csv(PREDICT_FILE)

print(f"Transaction records: {len(df_txn):,}")
print(f"Alert accounts:       {len(df_alert):,}")
print(f"Prediction accounts:  {len(df_predict):,}")

In [ ]:
# Normalize transaction amounts to TWD using the rates used in the competition pipeline.
currency_to_twd = {
    "TWD": 1.0, "USD": 32.0, "JPY": 0.22, "CNY": 4.4, "EUR": 35.0,
    "GBP": 41.0, "AUD": 21.0, "CAD": 23.0, "CHF": 36.0, "HKD": 4.1,
    "NZD": 19.0, "SEK": 3.0, "SGD": 23.0, "THB": 0.88, "ZAR": 1.8,
}

df_txn["txn_amt"] = pd.to_numeric(df_txn["txn_amt"], errors="coerce").fillna(0)
exchange_rate = df_txn["currency_type"].map(currency_to_twd).fillna(1.0)
df_txn["txn_amt_twd"] = df_txn["txn_amt"] * exchange_rate

df_txn["hour"] = (
    pd.to_datetime(df_txn["txn_time"], format="%H:%M:%S", errors="coerce")
    .dt.hour.fillna(0).astype(int)
)
df_txn["is_night"] = ((df_txn["hour"] >= 0) & (df_txn["hour"] < 6)).astype(int)

In [ ]:
# Represent every transaction from both account perspectives.
df_out = df_txn.rename(columns={
    "from_acct": "acct",
    "to_acct": "counterparty",
    "from_acct_type": "acct_type",
    "to_acct_type": "counterparty_type",
})
df_out["direction"] = "out"

df_in = df_txn.rename(columns={
    "to_acct": "acct",
    "from_acct": "counterparty",
    "to_acct_type": "acct_type",
    "from_acct_type": "counterparty_type",
})
df_in["direction"] = "in"

df_long = pd.concat([df_out, df_in], ignore_index=True)

columns = [
    "acct", "acct_type", "counterparty", "counterparty_type", "direction",
    "is_self_txn", "channel_type", "currency_type",
    "txn_amt", "txn_amt_twd", "txn_date", "txn_time", "hour", "is_night",
]
df_long = (
    df_long[columns]
    .sort_values(["acct", "txn_date"])
    .reset_index(drop=True)
)

print(f"Long-format records: {len(df_long):,}")

In [ ]:
# Last observed transaction date for every account.
df_acct_last = (
    df_long[["acct", "txn_date"]]
    .groupby("acct", as_index=False)["txn_date"].max()
    .rename(columns={"txn_date": "last_txn_day"})
)

# Positive accounts are defined by the alert table.
df_ref_alert = df_alert[["acct", "event_date"]].copy()
df_ref_alert["ref_day_idx"] = df_ref_alert["event_date"]
df_ref_alert["label"] = 1
df_ref_alert["split"] = "train"
df_ref_alert = df_ref_alert[["acct", "ref_day_idx", "label", "split"]]

# Unlabeled training pool: accounts involved in E.SUN-to-E.SUN transactions,
# excluding known alerts and prediction accounts.
df_both_esun = df_txn[
    (df_txn["from_acct_type"] == 1) &
    (df_txn["to_acct_type"] == 1)
].copy()

both_esun_accounts = set(df_both_esun["from_acct"]) | set(df_both_esun["to_acct"])
alert_set = set(df_alert["acct"])
predict_set = set(df_predict["acct"])
normal_pool = both_esun_accounts - alert_set - predict_set

df_ref_unlabeled = (
    pd.DataFrame({"acct": sorted(normal_pool)})
    .merge(df_acct_last, on="acct", how="left")
    .assign(
        ref_day_idx=lambda d: d["last_txn_day"],
        label=0,
        split="train",
    )[["acct", "ref_day_idx", "label", "split"]]
)

# Prediction accounts.
df_ref_predict = (
    pd.DataFrame({"acct": df_predict["acct"].unique()})
    .merge(df_acct_last, on="acct", how="left")
    .assign(
        ref_day_idx=lambda d: d["last_txn_day"],
        label=-1,
        split="test",
    )[["acct", "ref_day_idx", "label", "split"]]
)

df_ref = pd.concat(
    [df_ref_alert, df_ref_unlabeled, df_ref_predict],
    ignore_index=True,
)

print(f"Positive train accounts:  {(df_ref['label'] == 1).sum():,}")
print(f"Unlabeled train accounts: {(df_ref['label'] == 0).sum():,}")
print(f"Prediction accounts:      {(df_ref['label'] == -1).sum():,}")

In [ ]:
# Leakage guard: alert accounts may only use transactions observed on or before
# their alert reference day. Unlabeled/test accounts use all available history.
df_merged = df_long.merge(df_ref, on="acct", how="inner")
alert_ref = df_ref[df_ref["label"] == 1][["acct", "ref_day_idx"]]

future_leaks = df_long.merge(alert_ref, on="acct", how="inner")
future_leaks = future_leaks[
    future_leaks["txn_date"] > future_leaks["ref_day_idx"]
][["acct", "txn_date"]].drop_duplicates()

if not future_leaks.empty:
    df_feature_base = df_merged.merge(
        future_leaks,
        on=["acct", "txn_date"],
        how="left",
        indicator=True,
    )
    df_feature_base = (
        df_feature_base[df_feature_base["_merge"] == "left_only"]
        .drop(columns="_merge")
    )
    print(f"Removed future alert transactions: {len(future_leaks):,}")
else:
    df_feature_base = df_merged.copy()
    print("No future alert transactions found.")

print(f"Feature-base records: {len(df_feature_base):,}")

In [ ]:
output_path = ARTIFACT_DIR / "df_feature.parquet"
df_feature_base.to_parquet(output_path, index=False)
print(f"Saved: {output_path}")